In [1]:
import os
import rasterio
import numpy as np
import tensorflow as tf
import keras

c:\Users\Kostas\anaconda3\envs\vit\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
image_directory = r"E:\S2Germany\Patches"

Check if the images have correct dimensions and bands

In [ ]:
# Check if the images have correct dimensions
for root, dirs, files in os.walk(image_directory):
    for file in files:
        if file.endswith('.tif'):
            image = rasterio.open(os.path.join(root, file))
            assert image.height == 134 and image.width == 134
            image.close

In [ ]:
# Check if the band count is 12
for root, dirs, files in os.walk(image_directory):
    for file in files:
        if file.endswith('.tif'):
            image = rasterio.open(os.path.join(root, file))
            assert image.count == 12
            image.close

Import the dataset as an array

In [3]:
def load_tiff_image(image_path):
    with rasterio.open(image_path) as src:
        return src.read()

In [ ]:
# # Try to fix dimension problem. This probably affects the accurracy, as it mixes the band order. I got 0 accuracy on 10 epochs. To be deleted
# def load_tiff_image(image_path):
#     with rasterio.open(image_path) as src:
#         image = src.read()
#     return np.transpose(image, (1, 2, 0))

In [4]:
import os

base_dir = image_directory
class_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

images = []
labels = []
for class_id in class_dirs:
    image_files = [f for f in os.listdir(os.path.join(base_dir, class_id)) if f.endswith('.tif')]
    for f in image_files:
        images.append(load_tiff_image(os.path.join(base_dir, class_id, f)))
        labels.append(int(class_id))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'E:\\S2Germany\\Patches'

In [6]:
import numpy as np
# Normalize values
images = [(image - np.min(image)) / (np.max(image) - np.min(image)) for image in images]

NameError: name 'images' is not defined

In [ ]:
images[100].shape

In [ ]:
print(type(images))

In [5]:
image_dataset = tf.data.Dataset.from_tensor_slices(images) 
label_dataset = tf.data.Dataset.from_tensor_slices(labels) 
dataset = tf.data.Dataset.zip((image_dataset, label_dataset)) # Zipped together. They are a tuple

NameError: name 'images' is not defined

In [ ]:
print(type(dataset))

In [ ]:
dataset.cardinality().numpy()

In [ ]:
from tensorflow.keras import layers

# Architecture
model = tf.keras.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(12, 134, 134)))  # Adjusted input shape
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])


batch_size = 8
epochs = 50

# Shuffle and batch the dataset
#dataset = dataset.shuffle(len(images)).batch(batch_size)

# Train the model
model.fit(dataset, epochs=epochs)
